In [1]:
#FILE IMPORT

import pandas as pd

# File save into a pandas dataframe for the easier next steps.
df_train = pd.read_csv("Documents/Datasets/Predict_calorie_expenditure/train.csv")
df_test= pd.read_csv("Documents/Datasets/Predict_calorie_expenditure/test.csv")

In [2]:
# TEST

# Check the reading step.
if not df_train.empty:
    print("The train dataset ran successfully:")
    print(df_train.columns)
else:
    print("The train dataset ran incorrectly :(")

if not df_test.empty:
    print("\nThe test dataset ran successfully:")
    print(df_test.columns)
else:
    print("\nThe test dataset ran incorrectly :(")

# Check the rows to understand the topic.
df_train.head()

The train dataset ran successfully:
Index(['id', 'Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate',
       'Body_Temp', 'Calories'],
      dtype='object')

The test dataset ran successfully:
Index(['id', 'Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate',
       'Body_Temp'],
      dtype='object')


,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


In [3]:
# IMPORTS
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

In [4]:
# DATA CLEANING
# Replace the values of sex from text type to number type.
df_train["Sex"] = df_train["Sex"].replace({"male": 1, "female": 0})
df_test["Sex"] = df_test["Sex"].replace({"male": 1, "female": 0})

C:\Users\Ben\AppData\Local\Temp\ipykernel_13136\3216570150.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_train["Sex"] = df_train["Sex"].replace({"male": 1, "female": 0})
C:\Users\Ben\AppData\Local\Temp\ipykernel_13136\3216570150.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_test["Sex"] = df_test["Sex"].replace({"male": 1, "female": 0})


In [5]:
# EXTRA CALCULATIONS
# Go ahead all over the df_train and df_test and write extra calculate columns.
for df in [df_train, df_test]:
    df["BMI"] = df["Weight"] / ((df["Height"] / 100) ** 2)
    df["HR_x_Duration"] = df["Heart_Rate"] * df["Duration"]
    df["Temp_x_Duration"] = df["Body_Temp"] * df["Duration"]
    df["Age_x_Duration"] = df["Age"] * df["Duration"]

In [6]:
# DATA FEATURES AND GOAL PIN
features = [
    "Sex", "Age", "Height", "Weight", "Duration",
    "Heart_Rate", "Body_Temp", "BMI",
    "HR_x_Duration", "Temp_x_Duration", "Age_x_Duration"
]

# Pin the train and the goal header
X = df_train[features]
y = df_train["Calories"]
# Pin only train header and the algortithms will guess out the goal values
X_test = df_test[features]

In [7]:
# TRAIN PREPARATION

# I work only the train set and I split it for 2 part.
# First part: The real training -> I know the features and the goal's value.
# Second part: Test the model which I trained before.
# test_size=0.2 means -> 80% train, 20% test
# random_state=42 means -> I mixed the rows to a better training.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# I expect a rational number result -> LinearRegression
model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression()
)

In [8]:
# TRAINING START

model.fit(X_train, y_train)

# Predict train goal's value which has already known and check it.
val_pred = model.predict(X_val)
val_pred = np.clip(val_pred, 0, None)

In [9]:
# CHECK THE ERROR OF RESULTS

# Root Mean Squared Logarithmic Error
## When the rates are more important than the differences.
rmsle = np.sqrt(mean_squared_log_error(y_val, val_pred))
# Mean Absolute Error
## How many calories is the system wrong.
mae = mean_absolute_error(y_val, val_pred)
# Root Mean Square Error
## It also measures the error in calories, but it penalizes large errors more severely because it first squares the differences.
rmse = np.sqrt(mean_squared_error(y_val, val_pred))

In [10]:
# DISPLAY

print("RMSLE:", rmsle)
print("MAE:", mae)
print("RMSE:", rmse)

RMSLE: 0.0753922232039488
MAE: 2.1910648619240063
RMSE: 3.688458454112095


In [11]:
# USE THE READY MODEL

model.fit(X, y)

test_pred = model.predict(X_test)
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({
    "id": df_test["id"],
    "Calories": test_pred
})

In [12]:
# SAVE THE PREDICTION INTO A .CSV FILE

submission.to_csv("calories_predictions.csv", index=False)
submission.head()


,id,Calories
0,750000,26.898628
1,750001,107.524968
2,750002,86.766366
3,750003,126.105736
4,750004,75.817156
